## Build NHDPlus Grids

In [ ]:
import os,sys,requests,zipfile,shutil;
from osgeo import gdal;

sys.path.append(os.path.join(os.path.expanduser('~'),'notebooks'));
import common;

lddk = os.path.join(os.path.expanduser('~'),'loading_dock');
lddk = r'/glacier/grids_2026';


In [ ]:
def check_link(url):
    try:
        response = requests.head(url,allow_redirects=True,timeout=5);
        if response.status_code != 200:
            print(f"Failed: {url} returned status code {response.status_code}.");
            
    except requests.exceptions.RequestException as e:
        print(f"Error: Could not connect to {url}. {e}");

def download_link(url,target):
    response = requests.get(url,stream=True);

    with open(target,"wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk);


### Grid attributes

* **gid** integer value representing a specific grid
* **url** source url
* **srid** the srid group to which the grid belongs, some grids may be incorrectly projected
* **priority** number used to sort grids for mosaicking
* **wrongsrid** incorrect srid of the grid needing warping to correct srid group
* **gridsize** expected grid size used in warp, for Alaska this is 5 meters.
* **fixsrid** boolean to force raster projection data to match EPSG code

NHDPlus grids vary in quality where mainly its the Alaska grids that are borked.  The above attributes allow for automated actions to prepare these not-so-hot grids for mosaicking.

As an example, NHDPlus HR grid 19060505 is both incorrectly projected and has a definition of that incorrect projection not recognized by GDAL.  The **fixsrid** flag first forces the projection to 5070 and then the **wrongsrid** flag mosaicks the raster to 3338.

In [ ]:
check_mr_links = False;
dwnld_mr_links = False;

mr_grid_size = {
     5070:  30
    ,26904: 10
    ,32161: 10
    ,32655: 10
    ,32702: 10
}

mr_grids = {
     "01a":  {"gid":7011,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusNE/NHDPlusV21_NE_01_01a_FdrFac_01.7z"}
    ,"02a":  {"gid":7021,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusMA/NHDPlusV21_MA_02_02a_FdrFac_01.7z"}
    ,"02b":  {"gid":7022,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusMA/NHDPlusV21_MA_02_02b_FdrFac_01.7z"}
    ,"03Na": {"gid":70311,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusSA/NHDPlus03N/NHDPlusV21_SA_03N_03a_FdrFac_01.7z"}
    ,"03Nb": {"gid":70312,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusSA/NHDPlus03N/NHDPlusV21_SA_03N_03b_FdrFac_01.7z"}
    ,"03Sc": {"gid":70323,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusSA/NHDPlus03S/NHDPlusV21_SA_03S_03c_FdrFac_01.7z"}
    ,"03Sd": {"gid":70324,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusSA/NHDPlus03S/NHDPlusV21_SA_03S_03d_FdrFac_01.7z"}
    ,"03We": {"gid":70331,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusSA/NHDPlus03W/NHDPlusV21_SA_03W_03e_FdrFac_01.7z"}
    ,"03Wf": {"gid":70332,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusSA/NHDPlus03W/NHDPlusV21_SA_03W_03f_FdrFac_01.7z"}
    ,"04a":  {"gid":7041,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusGL/NHDPlusV21_GL_04_04a_FdrFac_03.7z"}
    ,"04b":  {"gid":7042,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusGL/NHDPlusV21_GL_04_04b_FdrFac_03.7z"}
    ,"04c":  {"gid":7043,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusGL/NHDPlusV21_GL_04_04c_FdrFac_03.7z"}
    ,"04d":  {"gid":7044,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusGL/NHDPlusV21_GL_04_04d_FdrFac_03.7z"}
    ,"05a":  {"gid":7051,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusMS/NHDPlus05/NHDPlusV21_MS_05_05a_FdrFac_01.7z"}
    ,"05b":  {"gid":7052,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusMS/NHDPlus05/NHDPlusV21_MS_05_05b_FdrFac_01.7z"}
    ,"05c":  {"gid":7053,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusMS/NHDPlus05/NHDPlusV21_MS_05_05c_FdrFac_01.7z"}
    ,"05d":  {"gid":7054,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusMS/NHDPlus05/NHDPlusV21_MS_05_05d_FdrFac_01.7z"}
    ,"06a":  {"gid":7061,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusMS/NHDPlus06/NHDPlusV21_MS_06_06a_FdrFac_03.7z"}
    ,"07a":  {"gid":7071,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusMS/NHDPlus07/NHDPlusV21_MS_07_07a_FdrFac_01.7z"}
    ,"07b":  {"gid":7072,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusMS/NHDPlus07/NHDPlusV21_MS_07_07b_FdrFac_01.7z"}
    ,"07c":  {"gid":7073,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusMS/NHDPlus07/NHDPlusV21_MS_07_07c_FdrFac_01.7z"}
    ,"08a":  {"gid":7081,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusMS/NHDPlus08/NHDPlusV21_MS_08_08a_FdrFac_01.7z"}
    ,"08b":  {"gid":7082,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusMS/NHDPlus08/NHDPlusV21_MS_08_08b_FdrFac_01.7z"}
    ,"03g":  {"gid":7083,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusMS/NHDPlus08/NHDPlusV21_MS_08_03g_FdrFac_01.7z"}
    ,"09a":  {"gid":7091,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusSR/NHDPlusV21_SR_09_09a_FdrFac_01.7z"}
    ,"10La": {"gid":71011,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusMS/NHDPlus10L/NHDPlusV21_MS_10L_10a_FdrFac_01.7z"}
    ,"10Lb": {"gid":71012,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusMS/NHDPlus10L/NHDPlusV21_MS_10L_10b_FdrFac_01.7z"}
    ,"10Lc": {"gid":71013,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusMS/NHDPlus10L/NHDPlusV21_MS_10L_10c_FdrFac_01.7z"}
    ,"10Ld": {"gid":71014,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusMS/NHDPlus10L/NHDPlusV21_MS_10L_10d_FdrFac_01.7z"}
    ,"10Ue": {"gid":71025,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusMS/NHDPlus10U/NHDPlusV21_MS_10U_10e_FdrFac_02.7z"}
    ,"10Uf": {"gid":71026,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusMS/NHDPlus10U/NHDPlusV21_MS_10U_10f_FdrFac_02.7z"}
    ,"10Ug": {"gid":71027,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusMS/NHDPlus10U/NHDPlusV21_MS_10U_10g_FdrFac_02.7z"}
    ,"10Uh": {"gid":71028,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusMS/NHDPlus10U/NHDPlusV21_MS_10U_10h_FdrFac_02.7z"}
    ,"10Ui": {"gid":71029,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusMS/NHDPlus10U/NHDPlusV21_MS_10U_10i_FdrFac_02.7z"}
    ,"11a":  {"gid":7111,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusMS/NHDPlus11/NHDPlusV21_MS_11_11a_FdrFac_01.7z"}
    ,"11b":  {"gid":7112,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusMS/NHDPlus11/NHDPlusV21_MS_11_11b_FdrFac_01.7z"}
    ,"11c":  {"gid":7113,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusMS/NHDPlus11/NHDPlusV21_MS_11_11c_FdrFac_01.7z"}
    ,"11d":  {"gid":7114,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusMS/NHDPlus11/NHDPlusV21_MS_11_11d_FdrFac_01.7z"}
    ,"12a":  {"gid":7121,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusTX/NHDPlusV21_TX_12_12a_FdrFac_01.7z"}
    ,"12b":  {"gid":7122,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusTX/NHDPlusV21_TX_12_12b_FdrFac_01.7z"}
    ,"12c":  {"gid":7123,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusTX/NHDPlusV21_TX_12_12c_FdrFac_01.7z"}
    ,"12d":  {"gid":7124,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusTX/NHDPlusV21_TX_12_12d_FdrFac_01.7z"}
    ,"13a":  {"gid":7131,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusRG/NHDPlusV21_RG_13_13a_FdrFac_01.7z"}
    ,"13b":  {"gid":7132,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusRG/NHDPlusV21_RG_13_13b_FdrFac_01.7z"}
    ,"13c":  {"gid":7133,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusRG/NHDPlusV21_RG_13_13c_FdrFac_01.7z"}
    ,"13d":  {"gid":7134,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusRG/NHDPlusV21_RG_13_13d_FdrFac_01.7z"}
    ,"14a":  {"gid":7141,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusCO/NHDPlus14/NHDPlusV21_CO_14_14a_FdrFac_01.7z"}
    ,"14b":  {"gid":7142,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusCO/NHDPlus14/NHDPlusV21_CO_14_14b_FdrFac_01.7z"}
    ,"15a":  {"gid":7151,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusCO/NHDPlus15/NHDPlusV21_CO_15_15a_FdrFac_01.7z"}
    ,"15b":  {"gid":7152,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusCO/NHDPlus15/NHDPlusV21_CO_15_15b_FdrFac_01.7z"}
    ,"16a":  {"gid":7161,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusGB/NHDPlusV21_GB_16_16a_FdrFac_01.7z"}
    ,"16b":  {"gid":7162,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusGB/NHDPlusV21_GB_16_16b_FdrFac_01.7z"}
    ,"17a":  {"gid":7171,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusPN/NHDPlusV21_PN_17_17a_FdrFac_01.7z"}
    ,"17b":  {"gid":7172,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusPN/NHDPlusV21_PN_17_17b_FdrFac_01.7z"}
    ,"17c":  {"gid":7173,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusPN/NHDPlusV21_PN_17_17c_FdrFac_01.7z"}
    ,"17d":  {"gid":7174,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusPN/NHDPlusV21_PN_17_17d_FdrFac_01.7z"}
    ,"18a":  {"gid":7181,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusCA/NHDPlusV21_CA_18_18a_FdrFac_01.7z"}
    ,"18b":  {"gid":7182,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusCA/NHDPlusV21_CA_18_18b_FdrFac_01.7z"}
    ,"18c":  {"gid":7183,"srid":5070,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusCA/NHDPlusV21_CA_18_18c_FdrFac_01.7z"}
    ,"20a":  {"gid":7201,"srid":26904,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusHI/NHDPlusV21_HI_20_20a_FdrFac_01.7z"}
    ,"20b":  {"gid":7202,"srid":26904,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusHI/NHDPlusV21_HI_20_20b_FdrFac_01.7z"}
    ,"20c":  {"gid":7203,"srid":26904,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusHI/NHDPlusV21_HI_20_20c_FdrFac_01.7z"}
    ,"20d":  {"gid":7204,"srid":26904,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusHI/NHDPlusV21_HI_20_20d_FdrFac_01.7z"}
    ,"20e":  {"gid":7205,"srid":26904,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusHI/NHDPlusV21_HI_20_20e_FdrFac_01.7z"}
    ,"20f":  {"gid":7206,"srid":26904,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusHI/NHDPlusV21_HI_20_20f_FdrFac_01.7z"}
    ,"20g":  {"gid":7207,"srid":26904,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusHI/NHDPlusV21_HI_20_20g_FdrFac_01.7z"}
    ,"20h":  {"gid":7208,"srid":26904,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusHI/NHDPlusV21_HI_20_20h_FdrFac_01.7z"}
    ,"21a":  {"gid":7211,"srid":32161,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusCI/NHDPlusV21_CI_21_21a_FdrFac_01.7z"}
    ,"21b":  {"gid":7212,"srid":32161,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusCI/NHDPlusV21_CI_21_21b_FdrFac_01.7z"}
    ,"21c":  {"gid":7213,"srid":32161,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusCI/NHDPlusV21_CI_21_21c_FdrFac_01.7z"}
    ,"22GUa":{"gid":7221,"srid":32655,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusPI/NHDPlus22GU/NHDPlusV21_PI_22GU_22a_FdrFac_01.7z"}
    ,"22MPb":{"gid":7222,"srid":32655,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusPI/NHDPlus22MP/NHDPlusV21_PI_22MP_22b_FdrFac_01.7z"}
    ,"22ASc":{"gid":7223,"srid":32702,"priority":5,"rastertype":"esrigrid"
              ,"url":"https://dmap-data-commons-ow.s3.amazonaws.com/NHDPlusV21/Data/NHDPlusPI/NHDPlus22AS/NHDPlusV21_PI_22AS_22c_FdrFac_01.7z"}
};

if check_mr_links is True:
    for k,v in mr_grids.items():
        check_link(v['url']);

if dwnld_mr_links is True:
    for k,v in mr_grids.items():
        download_link(
             v['url']
            ,os.path.join(lddk,'mr',os.path.basename(v['url']))
        );
    

In [ ]:
check_hr_r1_links = False;
dwnld_hr_r1_links = False;

hr_r1_grid_size = {
     3338:  5
    ,5070:  10
    ,26904: 10
    ,32161: 10
    ,32655: 10.233
    ,32702: 10.18614404967
}

hr_r1_grids = {
    "0101":  {"gid":80101,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0101_HU4_RASTER.7z"}
   ,"0102":  {"gid":80102,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0102_HU4_RASTER.7z"}
   ,"0103":  {"gid":80103,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0103_HU4_RASTER.7z"}
   ,"0104":  {"gid":80104,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0104_HU4_RASTER.7z"}
   ,"0105":  {"gid":80105,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0105_HU4_RASTER.7z"}
   ,"0106":  {"gid":80106,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0106_HU4_RASTER.7z"}
   ,"0107":  {"gid":80107,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0107_HU4_RASTER.7z"}
   ,"0108":  {"gid":80108,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0108_HU4_RASTER.7z"}
   ,"0109":  {"gid":80109,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0109_HU4_RASTER.7z"}
   ,"0110":  {"gid":80110,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0110_HU4_RASTER.7z"}
   ,"0202":  {"gid":80202,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0202_HU4_RASTER.7z"}
   ,"0203":  {"gid":80203,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0203_HU4_RASTER.7z"}
   ,"0204":  {"gid":80204,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0204_HU4_RASTER.7z"}
   ,"0205":  {"gid":80205,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0205_HU4_RASTER.7z"}
   ,"0206":  {"gid":80206,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0206_HU4_RASTER.7z"}
   ,"0207":  {"gid":80207,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0207_HU4_RASTER.7z"}
   ,"0208":  {"gid":80208,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0208_HU4_RASTER.7z"}
   ,"0301":  {"gid":80301,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0301_HU4_RASTER.7z"}
   ,"0302":  {"gid":80302,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0302_HU4_RASTER.7z"}
   ,"0303":  {"gid":80303,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0303_HU4_RASTER.7z"}
   ,"0304":  {"gid":80304,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0304_HU4_RASTER.7z"}
   ,"0305":  {"gid":80305,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0305_HU4_RASTER.7z"}
   ,"0306":  {"gid":80306,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0306_HU4_RASTER.7z"}
   ,"0307":  {"gid":80307,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0307_HU4_RASTER.7z"}
   ,"0308":  {"gid":80308,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0308_HU4_RASTER.7z"}
   ,"0309":  {"gid":80309,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0309_HU4_RASTER.7z"}
   ,"0310":  {"gid":80310,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0310_HU4_RASTER.7z"}
   ,"0311":  {"gid":80311,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0311_HU4_RASTER.7z"}
   ,"0312":  {"gid":80312,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0312_HU4_RASTER.7z"}
   ,"0313":  {"gid":80313,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0313_HU4_RASTER.7z"}
   ,"0314":  {"gid":80314,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0314_HU4_RASTER.7z"}
   ,"0315":  {"gid":80315,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0315_HU4_RASTER.7z"}
   ,"0316":  {"gid":80316,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0316_HU4_RASTER.7z"}
   ,"0317":  {"gid":80317,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0317_HU4_RASTER.7z"}
   ,"0318":  {"gid":80318,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0318_HU4_RASTER.7z"}
   ,"0401":  {"gid":80401,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0401_HU4_RASTER.7z"}
   ,"0402":  {"gid":80402,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0402_HU4_RASTER.7z"}
   ,"0403":  {"gid":80403,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0403_HU4_RASTER.7z"}
   ,"0404":  {"gid":80404,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0404_HU4_RASTER.7z"}
   ,"0405":  {"gid":80405,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0405_HU4_RASTER.7z"}
   ,"0406":  {"gid":80406,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0406_HU4_RASTER.7z"}
   ,"0407":  {"gid":80407,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0407_HU4_RASTER.7z"}
   ,"0408":  {"gid":80408,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0408_HU4_RASTER.7z"}
   ,"0409":  {"gid":80409,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0409_HU4_RASTER.7z"}
   ,"0410":  {"gid":80410,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0410_HU4_RASTER.7z"}
   ,"0411":  {"gid":80411,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0411_HU4_RASTER.7z"}
   ,"0412":  {"gid":80412,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0412_HU4_RASTER.7z"}
   ,"0413":  {"gid":80413,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0413_HU4_RASTER.7z"}
   ,"0414":  {"gid":80414,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0414_HU4_RASTER.7z"}
   ,"0416":  {"gid":80416,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0416_HU4_RASTER.7z"}
   ,"0417":  {"gid":80417,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0417_HU4_RASTER.7z"}
   ,"0418":  {"gid":80418,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0418_HU4_RASTER.7z"}
   ,"0418i": {"gid":804181,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0418i_HU4_RASTER.7z"}
   ,"0419":  {"gid":80419,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0419_HU4_RASTER.7z"}
   ,"0419i": {"gid":804191,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0419i_HU4_RASTER.7z"}
   ,"0420":  {"gid":80420,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0420_HU4_RASTER.7z"}
   ,"0421":  {"gid":80421,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0421_HU4_RASTER.7z"}
   ,"0422":  {"gid":80422,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0422_HU4_RASTER.7z"}
   ,"0423":  {"gid":80423,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0423_HU4_RASTER.7z"}
   ,"0424":  {"gid":80424,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0424_HU4_RASTER.7z"}
   ,"0424i": {"gid":804241,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0424i_HU4_RASTER.7z"}
   ,"0425":  {"gid":80425,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0425_HU4_RASTER.7z"}
   ,"0426":  {"gid":80426,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0426_HU4_RASTER.7z"}
   ,"0426i": {"gid":804261,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0426i_HU4_RASTER.7z"}
   ,"0427":  {"gid":80427,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0427_HU4_RASTER.7z"}
   ,"0428":  {"gid":80428,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0428_HU4_RASTER.7z"}
   ,"0428i": {"gid":804281,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0428i_HU4_RASTER.7z"}
   ,"0429":  {"gid":80429,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0429_HU4_RASTER.7z"}
   ,"0430":  {"gid":80430,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0430_HU4_RASTER.7z"}
   ,"0431":  {"gid":80431,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0431_HU4_RASTER.7z"}
   ,"0432":  {"gid":80432,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0432_HU4_RASTER.7z"}
   ,"0433":  {"gid":80433,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0433_HU4_RASTER.7z"}
   ,"0501":  {"gid":80501,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0501_HU4_RASTER.7z"}
   ,"0502":  {"gid":80502,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0502_HU4_RASTER.7z"}
   ,"0503":  {"gid":80503,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0503_HU4_RASTER.7z"}
   ,"0504":  {"gid":80504,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0504_HU4_RASTER.7z"}
   ,"0505":  {"gid":80505,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0505_HU4_RASTER.7z"}
   ,"0506":  {"gid":80506,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0506_HU4_RASTER.7z"}
   ,"0507":  {"gid":80507,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0507_HU4_RASTER.7z"}
   ,"0508":  {"gid":80508,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0508_HU4_RASTER.7z"}
   ,"0509":  {"gid":80509,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0509_HU4_RASTER.7z"}
   ,"0510":  {"gid":80510,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0510_HU4_RASTER.7z"}
   ,"0511":  {"gid":80511,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0511_HU4_RASTER.7z"}
   ,"0512":  {"gid":80512,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0512_HU4_RASTER.7z"}
   ,"0513":  {"gid":80513,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0513_HU4_RASTER.7z"}
   ,"0514":  {"gid":80514,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0514_HU4_RASTER.7z"}
   ,"0601":  {"gid":80601,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0601_HU4_RASTER.7z"}
   ,"0602":  {"gid":80602,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0602_HU4_RASTER.7z"}
   ,"0603":  {"gid":80603,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0603_HU4_RASTER.7z"}
   ,"0604":  {"gid":80604,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0604_HU4_RASTER.7z"}
   ,"0701":  {"gid":80701,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0701_HU4_RASTER.7z"}
   ,"0702":  {"gid":80702,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0702_HU4_RASTER.7z"}
   ,"0703":  {"gid":80703,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0703_HU4_RASTER.7z"}
   ,"0704":  {"gid":80704,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0704_HU4_RASTER.7z"}
   ,"0705":  {"gid":80705,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0705_HU4_RASTER.7z"}
   ,"0706":  {"gid":80706,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0706_HU4_RASTER.7z"}
   ,"0707":  {"gid":80707,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0707_HU4_RASTER.7z"}
   ,"0708":  {"gid":80708,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0708_HU4_RASTER.7z"}
   ,"0709":  {"gid":80709,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0709_HU4_RASTER.7z"}
   ,"0710":  {"gid":80710,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0710_HU4_RASTER.7z"}
   ,"0711":  {"gid":80711,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0711_HU4_RASTER.7z"}
   ,"0712":  {"gid":80712,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0712_HU4_RASTER.7z"}
   ,"0713":  {"gid":80713,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0713_HU4_RASTER.7z"}
   ,"0714":  {"gid":80714,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0714_HU4_RASTER.7z"}
   ,"0801":  {"gid":80801,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0801_HU4_RASTER.7z"}
   ,"0802":  {"gid":80802,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0802_HU4_RASTER.7z"}
   ,"0803":  {"gid":80803,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0803_HU4_RASTER.7z"}
   ,"0804":  {"gid":80804,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0804_HU4_RASTER.7z"}
   ,"0805":  {"gid":80805,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0805_HU4_RASTER.7z"}
   ,"0806":  {"gid":80806,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0806_HU4_RASTER.7z"}
   ,"0807":  {"gid":80807,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0807_HU4_RASTER.7z"}
   ,"0808":  {"gid":80808,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0808_HU4_RASTER.7z"}
   ,"0809":  {"gid":80809,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0809_HU4_RASTER.7z"}
   ,"0901":  {"gid":80901,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0901_HU4_RASTER.7z"}
   ,"0902":  {"gid":80902,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0902_HU4_RASTER.7z"}
   ,"0903":  {"gid":80903,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0903_HU4_RASTER.7z"}
   ,"0904":  {"gid":80904,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0904_HU4_RASTER.7z"}
   ,"1002":  {"gid":81002,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1002_HU4_RASTER.7z"}
   ,"1003":  {"gid":81003,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1003_HU4_RASTER.7z"}
   ,"1004":  {"gid":81004,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1004_HU4_RASTER.7z"}
   ,"1005":  {"gid":81005,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1005_HU4_RASTER.7z"}
   ,"1006":  {"gid":81006,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1006_HU4_RASTER.7z"}
   ,"1007":  {"gid":81007,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1007_HU4_RASTER.7z"}
   ,"1008":  {"gid":81008,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1008_HU4_RASTER.7z"}
   ,"1009":  {"gid":81009,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1009_HU4_RASTER.7z"}
   ,"1010":  {"gid":81010,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1010_HU4_RASTER.7z"}
   ,"1011":  {"gid":81011,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1011_HU4_RASTER.7z"}
   ,"1012":  {"gid":81012,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1012_HU4_RASTER.7z"}
   ,"1013":  {"gid":81013,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1013_HU4_RASTER.7z"}
   ,"1014":  {"gid":81014,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1014_HU4_RASTER.7z"}
   ,"1015":  {"gid":81015,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1015_HU4_RASTER.7z"}
   ,"1016":  {"gid":81016,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1016_HU4_RASTER.7z"}
   ,"1017":  {"gid":81017,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1017_HU4_RASTER.7z"}
   ,"1018":  {"gid":81018,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1018_HU4_RASTER.7z"}
   ,"1019":  {"gid":81019,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1019_HU4_RASTER.7z"}
   ,"1020":  {"gid":81020,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1020_HU4_RASTER.7z"}
   ,"1021":  {"gid":81021,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1021_HU4_RASTER.7z"}
   ,"1022":  {"gid":81022,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1022_HU4_RASTER.7z"}
   ,"1023":  {"gid":81023,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1023_HU4_RASTER.7z"}
   ,"1024":  {"gid":81024,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1024_HU4_RASTER.7z"}
   ,"1025":  {"gid":81025,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1025_HU4_RASTER.7z"}
   ,"1026":  {"gid":81026,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1026_HU4_RASTER.7z"}
   ,"1027":  {"gid":81027,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1027_HU4_RASTER.7z"}
   ,"1028":  {"gid":81028,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1028_HU4_RASTER.7z"}
   ,"1029":  {"gid":81029,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1029_HU4_RASTER.7z"}
   ,"1030":  {"gid":81030,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1030_HU4_RASTER.7z"}
   ,"1101":  {"gid":81101,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1101_HU4_RASTER.7z"}
   ,"1102":  {"gid":81102,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1102_HU4_RASTER.7z"}
   ,"1103":  {"gid":81103,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1103_HU4_RASTER.7z"}
   ,"1104":  {"gid":81104,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1104_HU4_RASTER.7z"}
   ,"1105":  {"gid":81105,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1105_HU4_RASTER.7z"}
   ,"1106":  {"gid":81106,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1106_HU4_RASTER.7z"}
   ,"1107":  {"gid":81107,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1107_HU4_RASTER.7z"}
   ,"1108":  {"gid":81108,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1108_HU4_RASTER.7z"}
   ,"1109":  {"gid":81109,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1109_HU4_RASTER.7z"}
   ,"1110":  {"gid":81110,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1110_HU4_RASTER.7z"}
   ,"1111":  {"gid":81111,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1111_HU4_RASTER.7z"}
   ,"1112":  {"gid":81112,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1112_HU4_RASTER.7z"}
   ,"1113":  {"gid":81113,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1113_HU4_RASTER.7z"}
   ,"1114":  {"gid":81114,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1114_HU4_RASTER.7z"}
   ,"1201":  {"gid":81201,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1201_HU4_RASTER.7z"}
   ,"1202":  {"gid":81202,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1202_HU4_RASTER.7z"}
   ,"1203":  {"gid":81203,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1203_HU4_RASTER.7z"}
   ,"1204":  {"gid":81204,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1204_HU4_RASTER.7z"}
   ,"1205":  {"gid":81205,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1205_HU4_RASTER.7z"}
   ,"1206":  {"gid":81206,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1206_HU4_RASTER.7z"}
   ,"1207":  {"gid":81207,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1207_HU4_RASTER.7z"}
   ,"1208":  {"gid":81208,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1208_HU4_RASTER.7z"}
   ,"1209":  {"gid":81209,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1209_HU4_RASTER.7z"}
   ,"1210":  {"gid":81210,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1210_HU4_RASTER.7z"}
   ,"1211":  {"gid":81211,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1211_HU4_RASTER.7z"}
   ,"1301":  {"gid":81301,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1301_HU4_RASTER.7z"}
   ,"1302":  {"gid":81302,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1302_HU4_RASTER.7z"}
   ,"1303":  {"gid":81303,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1303_HU4_RASTER.7z"}
   ,"1304":  {"gid":81304,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1304_HU4_RASTER.7z"}
   ,"1305":  {"gid":81305,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1305_HU4_RASTER.7z"}
   ,"1306":  {"gid":81306,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1306_HU4_RASTER.7z"}
   ,"1307":  {"gid":81307,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1307_HU4_RASTER.7z"}
   ,"1308":  {"gid":81308,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1308_HU4_RASTER.7z"}
   ,"1309":  {"gid":81309,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1309_HU4_RASTER.7z"}
   ,"1401":  {"gid":81401,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1401_HU4_RASTER.7z"}
   ,"1402":  {"gid":81402,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1402_HU4_RASTER.7z"}
   ,"1403":  {"gid":81403,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1403_HU4_RASTER.7z"}
   ,"1404":  {"gid":81404,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1404_HU4_RASTER.7z"}
   ,"1405":  {"gid":81405,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1405_HU4_RASTER.7z"}
   ,"1406":  {"gid":81406,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1406_HU4_RASTER.7z"}
   ,"1407":  {"gid":81407,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1407_HU4_RASTER.7z"}
   ,"1408":  {"gid":81408,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1408_HU4_RASTER.7z"}
   ,"1501":  {"gid":81501,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1501_HU4_RASTER.7z"}
   ,"1502":  {"gid":81502,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1502_HU4_RASTER.7z"}
   ,"1503":  {"gid":81503,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1503_HU4_RASTER.7z"}
   ,"1504":  {"gid":81504,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1504_HU4_RASTER.7z"}
   ,"1505":  {"gid":81505,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1505_HU4_RASTER.7z"}
   ,"1506":  {"gid":81506,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1506_HU4_RASTER.7z"}
   ,"1507":  {"gid":81507,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1507_HU4_RASTER.7z"}
   ,"1508":  {"gid":81508,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1508_HU4_RASTER.7z"}
   ,"1601":  {"gid":81601,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1601_HU4_RASTER.7z"}
   ,"1602":  {"gid":81602,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1602_HU4_RASTER.7z"}
   ,"1603":  {"gid":81603,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1603_HU4_RASTER.7z"}
   ,"1604":  {"gid":81604,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1604_HU4_RASTER.7z"}
   ,"1605":  {"gid":81605,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1605_HU4_RASTER.7z"}
   ,"1606":  {"gid":81606,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1606_HU4_RASTER.7z"}
   ,"1701":  {"gid":81701,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1701_HU4_RASTER.7z"}
   ,"1702":  {"gid":81702,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1702_HU4_RASTER.7z"}
   ,"1703":  {"gid":81703,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1703_HU4_RASTER.7z"}
   ,"1704":  {"gid":81704,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1704_HU4_RASTER.7z"}
   ,"1705":  {"gid":81705,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1705_HU4_RASTER.7z"}
   ,"1706":  {"gid":81706,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1706_HU4_RASTER.7z"}
   ,"1707":  {"gid":81707,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1707_HU4_RASTER.7z"}
   ,"1708":  {"gid":81708,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1708_HU4_RASTER.7z"}
   ,"1709":  {"gid":81709,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1709_HU4_RASTER.7z"}
   ,"1710":  {"gid":81710,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1710_HU4_RASTER.7z"}
   ,"1711":  {"gid":81711,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1711_HU4_RASTER.7z"}
   ,"1712":  {"gid":81712,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1712_HU4_RASTER.7z"}
   ,"1801":  {"gid":81801,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1801_HU4_RASTER.7z"}
   ,"1802":  {"gid":81802,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1802_HU4_RASTER.7z"}
   ,"1803":  {"gid":81803,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1803_HU4_RASTER.7z"}
   ,"1804":  {"gid":81804,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1804_HU4_RASTER.7z"}
   ,"1805":  {"gid":81805,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1805_HU4_RASTER.7z"}
   ,"1806":  {"gid":81806,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1806_HU4_RASTER.7z"}
   ,"1807":  {"gid":81807,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1807_HU4_RASTER.7z"}
   ,"1808":  {"gid":81808,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1808_HU4_RASTER.7z"}
   ,"1809":  {"gid":81809,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1809_HU4_RASTER.7z"}
   ,"1810":  {"gid":81810,"srid":5070,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_1810_HU4_RASTER.7z"}
   ,"19020101": {"gid":819020101,"srid":3338,"priority":5,"rastertype":"tiff"
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_19020101_HU8_RASTER.7z"}
   ,"19020102": {"gid":819020102,"srid":3338,"priority":5,"rastertype":"tiff"
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_19020102_HU8_RASTER.7z"}
   ,"19020103": {"gid":819020103,"srid":3338,"priority":5,"rastertype":"tiff"
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_19020103_HU8_RASTER.7z"}
   ,"19020104": {"gid":819020104,"srid":3338,"priority":5,"rastertype":"tiff"
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_19020104_HU8_RASTER.7z"}
   ,"19020202": {"gid":819020202,"srid":3338,"notes":"missing from beta downloads","priority":4,"rastertype":"tiff","badproj":5070,"gridsize":5
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_19020202_HU8_RASTER.zip"}
   ,"19020301": {"gid":819020301,"srid":3338,"priority":5,"rastertype":"tiff","badproj":5070,"gridsize":5
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_19020301_HU8_RASTER.7z"}
   ,"19020302": {"gid":819020302,"srid":3338,"priority":5,"rastertype":"tiff","badproj":5070,"gridsize":5
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_19020302_HU8_RASTER.7z"}
   ,"19020401": {"gid":819020401,"srid":3338,"priority":5,"rastertype":"tiff","forcesrid":3338
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_19020401_HU8_RASTER.7z"}
   ,"19020402": {"gid":819020402,"srid":3338,"priority":5,"rastertype":"tiff","forcesrid":3338
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_19020402_HU8_RASTER.7z"}
   ,"19020501": {"gid":819020501,"srid":3338,"priority":5,"rastertype":"tiff","forcesrid":3338
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_19020501_HU8_RASTER.7z"}
   ,"19020502": {"gid":819020502,"srid":3338,"priority":5,"rastertype":"tiff","forcesrid":3338
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_19020502_HU8_RASTER.7z"}
   ,"19020503": {"gid":819020503,"srid":3338,"priority":5,"rastertype":"tiff","forcesrid":3338
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_19020503_HU8_RASTER.7z"}
   ,"19020504": {"gid":819020504,"srid":3338,"priority":5,"rastertype":"tiff","forcesrid":3338
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_19020504_HU8_RASTER.7z"}
   ,"19020505": {"gid":819020505,"srid":3338,"priority":5,"rastertype":"tiff","forcesrid":3338
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_19020505_HU8_RASTER.7z"}
   ,"19020601": {"gid":819020601,"srid":3338,"priority":5,"rastertype":"tiff","badproj":5070,"gridsize":5
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_19020601_HU8_RASTER.7z"}
   ,"19020602": {"gid":819020602,"srid":3338,"priority":5,"rastertype":"tiff","badproj":5070,"gridsize":5
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_19020602_HU8_RASTER.7z"}
   ,"19020800": {"gid":819020800,"srid":3338,"priority":5,"rastertype":"tiff","badproj":5070,"gridsize":5
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_19020800_HU8_RASTER.7z"}
   ,"19060102": {"gid":819060102,"srid":3338,"priority":5,"rastertype":"tiff"
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_19060102_HU8_RASTER.7z"}
   ,"19070402": {"gid":819070402,"srid":3338,"priority":5,"rastertype":"tiff"
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_19070402_HU8_RASTER.7z"}
   ,"19080301": {"gid":819080301,"srid":3338,"priority":5,"rastertype":"tiff"
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_19080301_HU8_RASTER.7z"}
   ,"19080305": {"gid":819080305,"srid":3338,"priority":5,"rastertype":"tiff"
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_19080305_HU8_RASTER.7z"}
   ,"2001":  {"gid":82001,"srid":26904,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_2001_HU4_RASTER.7z"}
   ,"2002":  {"gid":82002,"srid":26904,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_2002_HU4_RASTER.7z"}
   ,"2003":  {"gid":82003,"srid":26904,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_2003_HU4_RASTER.7z"}
   ,"2004":  {"gid":82004,"srid":26904,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_2004_HU4_RASTER.7z"}
   ,"2005":  {"gid":82005,"srid":26904,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_2005_HU4_RASTER.7z"}
   ,"2006":  {"gid":82006,"srid":26904,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_2006_HU4_RASTER.7z"}
   ,"2007":  {"gid":82007,"srid":26904,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_2007_HU4_RASTER.7z"}
   ,"2008":  {"gid":82008,"srid":26904,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_2008_HU4_RASTER.7z"}
   ,"2101":  {"gid":82101,"srid":32161,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_2101_HU4_RASTER.7z"}
   ,"2102":  {"gid":82102,"srid":32161,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_2102_HU4_RASTER.7z"}
   ,"2201":  {"gid":82201,"srid":32655,"priority":5,"rastertype":"tiff","badproj":32655,"gridsize":10.233
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_2201_HU4_RASTER.7z"}
   ,"2202":  {"gid":82202,"srid":32655,"priority":5,"rastertype":"tiff","badproj":32655,"gridsize":10.233
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_2202_HU4_RASTER.7z"}
   ,"2203":  {"gid":82203,"srid":32702,"priority":5,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_2203_HU4_RASTER.7z"}
};

if check_hr_r1_links is True:
    for k,v in hr_r1_grids.items():
        check_link(v['url']);

if dwnld_hr_r1_links is True:
    for k,v in hr_r1_grids.items():
        download_link(
             v['url']
            ,os.path.join(lddk,'hr_r1',os.path.basename(v['url']))
        );


In [ ]:
check_hr_r2_links = False;
dwnld_hr_r2_links = False;

hr_r2_grid_size = {
     3338:  5
    ,5070:  10
    ,26904: 10
    ,32161: 10
    ,32655: 10.233
    ,32702: 10.18614404967
}

hr_r2_grids = {
    "0101":  {"gid":90101,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0101_HU4_20220901_RASTER.zip","srid":5070,"priority":10}
   ,"0102":  {"gid":90102,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0102_HU4_20230713_RASTER.zip","srid":5070,"priority":10}
   ,"0103":  {"gid":90103,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0103_HU4_20230713_RASTER.zip","srid":5070,"priority":10}
   ,"0104":  {"gid":90104,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0104_HU4_20230713_RASTER.zip","srid":5070,"priority":10}
   ,"0105":  {"gid":90105,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0105_HU4_20230713_RASTER.zip","srid":5070,"priority":10}
   ,"0106":  {"gid":90106,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0106_HU4_20230713_RASTER.zip","srid":5070,"priority":10}
   ,"0107":  {"gid":90107,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0107_HU4_20230713_RASTER.zip","srid":5070,"priority":10}
   ,"0108":  {"gid":90108,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0108_HU4_20220324_RASTER.zip","srid":5070,"priority":10}
   ,"0109":  {"gid":90109,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0109_HU4_20230713_RASTER.zip","srid":5070,"priority":10}
   ,"0110":  {"gid":90110,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0110_HU4_20220323_RASTER.zip","srid":5070,"priority":10}
   ,"0202":  {"gid":90202,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0202_HU4_20220324_RASTER.zip","srid":5070,"priority":10}
   ,"0203":  {"gid":90203,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0203_HU4_20220323_RASTER.zip","srid":5070,"priority":10}
   ,"0204":  {"gid":90204,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0204_HU4_20220324_RASTER.zip","srid":5070,"priority":10}
   ,"0205":  {"gid":90205,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0205_HU4_20220323_RASTER.zip","srid":5070,"priority":10}
   ,"0206":  {"gid":90206,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0206_HU4_20220324_RASTER.zip","srid":5070,"priority":10}
   ,"0207":  {"gid":90207,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0207_HU4_20220324_RASTER.zip","srid":5070,"priority":10}
   ,"0208":  {"gid":90208,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0208_HU4_20220324_RASTER.zip","srid":5070,"priority":10}
   ,"0301":  {"gid":90301,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0301_HU4_RASTER.zip"}
   ,"0302":  {"gid":90302,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0302_HU4_RASTER.zip"}
   ,"0303":  {"gid":90303,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0303_HU4_RASTER.zip"}
   ,"0304":  {"gid":90304,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0304_HU4_RASTER.zip"}
   ,"0305":  {"gid":90305,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0305_HU4_RASTER.zip"}
   ,"0306":  {"gid":90306,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0306_HU4_RASTER.zip"}
   ,"0307":  {"gid":90307,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0307_HU4_RASTER.zip"}
   ,"0308":  {"gid":90308,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0308_HU4_RASTER.zip"}
   ,"0309":  {"gid":90309,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0309_HU4_RASTER.zip"}
   ,"0310":  {"gid":90310,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0310_HU4_RASTER.zip"}
   ,"0311":  {"gid":90311,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0311_HU4_RASTER.zip"}
   ,"0312":  {"gid":90312,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0312_HU4_RASTER.zip"}
   ,"0313":  {"gid":90313,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0313_HU4_RASTER.zip"}
   ,"0314":  {"gid":90314,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0314_HU4_RASTER.zip"}
   ,"0315":  {"gid":90315,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0315_HU4_RASTER.zip"}
   ,"0316":  {"gid":90316,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0316_HU4_RASTER.zip"}
   ,"0317":  {"gid":90317,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0317_HU4_RASTER.zip"}
   ,"0318":  {"gid":90318,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0318_HU4_RASTER.zip"}
   ,"0401":  {"gid":90401,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0401_HU4_RASTER.zip"}
   ,"0402":  {"gid":90402,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0402_HU4_RASTER.zip"}
   ,"0403":  {"gid":90403,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0403_HU4_RASTER.zip"}
   ,"0404":  {"gid":90404,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0404_HU4_RASTER.zip"}
   ,"0405":  {"gid":90405,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0405_HU4_RASTER.zip"}
   ,"0406":  {"gid":90406,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0406_HU4_RASTER.zip"}
   ,"0407":  {"gid":90407,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0407_HU4_RASTER.zip"}
   ,"0408":  {"gid":90408,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0408_HU4_RASTER.zip"}
   ,"0409":  {"gid":90409,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0409_HU4_RASTER.zip"}
   ,"0410":  {"gid":90410,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0410_HU4_RASTER.zip"}
   ,"0411":  {"gid":90411,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0411_HU4_RASTER.zip"}
   ,"0412":  {"gid":90412,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0412_HU4_RASTER.zip"}
   ,"0413":  {"gid":90413,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0413_HU4_RASTER.zip"}
   ,"0414":  {"gid":90414,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0414_HU4_RASTER.zip"}
   ,"0416":  {"gid":90416,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0416_HU4_RASTER.zip"}
   ,"0417":  {"gid":90417,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0417_HU4_RASTER.zip"}
   ,"0418":  {"gid":90418,"srid":5070,"priority":1,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0418_HU4_RASTER.zip"}
   ,"0418i": {"gid":904181,"srid":5070,"priority":2,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0418i_HU4_RASTER.zip"}
   ,"0419":  {"gid":90419,"srid":5070,"priority":1,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0419_HU4_RASTER.zip"}
   ,"0419i": {"gid":904191,"srid":5070,"priority":2,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0419i_HU4_RASTER.zip"}
   ,"0420":  {"gid":90420,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0420_HU4_RASTER.zip"}
   ,"0421":  {"gid":90421,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0421_HU4_RASTER.zip"}
   ,"0422":  {"gid":90422,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0422_HU4_RASTER.zip"}
   ,"0423":  {"gid":90423,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0423_HU4_RASTER.zip"}
   ,"0424":  {"gid":90424,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0424_HU4_RASTER.zip"}
   ,"0424i": {"gid":904241,"srid":5070,"priority":1,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0424i_HU4_RASTER.zip"}
   ,"0425":  {"gid":90425,"srid":5070,"priority":2,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0425_HU4_RASTER.zip"}
   ,"0426":  {"gid":90426,"srid":5070,"priority":1,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0426_HU4_RASTER.zip"}
   ,"0426i": {"gid":904261,"srid":5070,"priority":2,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0426i_HU4_RASTER.zip"}
   ,"0427":  {"gid":90427,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0427_HU4_RASTER.zip"}
   ,"0428":  {"gid":90428,"srid":5070,"priority":1,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0428_HU4_RASTER.zip"}
   ,"0428i": {"gid":904281,"srid":5070,"priority":2,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0428i_HU4_RASTER.zip"}
   ,"0429":  {"gid":90429,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0429_HU4_RASTER.zip"}
   ,"0430":  {"gid":90430,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0430_HU4_RASTER.zip"}
   ,"0431":  {"gid":90431,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0431_HU4_RASTER.zip"}
   ,"0432":  {"gid":90432,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0432_HU4_RASTER.zip"}
   ,"0433":  {"gid":90433,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0433_HU4_RASTER.zip"}
   ,"0501":  {"gid":90501,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0501_HU4_RASTER.zip"}
   ,"0502":  {"gid":90502,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0502_HU4_RASTER.zip"}
   ,"0503":  {"gid":90503,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0503_HU4_RASTER.zip"}
   ,"0504":  {"gid":90504,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0504_HU4_RASTER.zip"}
   ,"0505":  {"gid":90505,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0505_HU4_RASTER.zip"}
   ,"0506":  {"gid":90506,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0506_HU4_RASTER.zip"}
   ,"0507":  {"gid":90507,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0507_HU4_RASTER.zip"}
   ,"0508":  {"gid":90508,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0508_HU4_RASTER.zip"}
   ,"0509":  {"gid":90509,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0509_HU4_RASTER.zip"}
   ,"0510":  {"gid":90510,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0510_HU4_RASTER.zip"}
   ,"0511":  {"gid":90511,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0511_HU4_RASTER.zip"}
   ,"0512":  {"gid":90512,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0512_HU4_RASTER.zip"}
   ,"0513":  {"gid":90513,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0513_HU4_RASTER.zip"}
   ,"0514":  {"gid":90514,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0514_HU4_RASTER.zip"}
   ,"0601":  {"gid":90601,"srid":5070,"notes":"r2 uses beta 06","priority":1,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0601_HU4_RASTER.7z"}
   ,"0602":  {"gid":90602,"srid":5070,"notes":"r2 uses beta 06","priority":1,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0602_HU4_RASTER.7z"}
   ,"0603":  {"gid":90603,"srid":5070,"notes":"r2 uses beta 06","priority":1,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0603_HU4_RASTER.7z"}
   ,"0604":  {"gid":90604,"srid":5070,"notes":"r2 uses beta 06","priority":1,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/Beta/GDB/NHDPLUS_H_0604_HU4_RASTER.7z"}
   ,"0701":  {"gid":90701,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0701_HU4_RASTER.zip"}
   ,"0702":  {"gid":90702,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0702_HU4_RASTER.zip"}
   ,"0703":  {"gid":90703,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0703_HU4_RASTER.zip"}
   ,"0704":  {"gid":90704,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0704_HU4_RASTER.zip"}
   ,"0705":  {"gid":90705,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0705_HU4_RASTER.zip"}
   ,"0706":  {"gid":90706,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0706_HU4_RASTER.zip"}
   ,"0707":  {"gid":90707,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0707_HU4_RASTER.zip"}
   ,"0708":  {"gid":90708,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0708_HU4_RASTER.zip"}
   ,"0709":  {"gid":90709,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0709_HU4_RASTER.zip"}
   ,"0710":  {"gid":90710,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0710_HU4_RASTER.zip"}
   ,"0711":  {"gid":90711,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0711_HU4_RASTER.zip"}
   ,"0712":  {"gid":90712,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0712_HU4_RASTER.zip"}
   ,"0713":  {"gid":90713,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0713_HU4_RASTER.zip"}
   ,"0714":  {"gid":90714,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0714_HU4_RASTER.zip"}
   ,"0801":  {"gid":90801,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0801_HU4_RASTER.zip"}
   ,"0802":  {"gid":90802,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0802_HU4_RASTER.zip"}
   ,"0803":  {"gid":90803,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0803_HU4_RASTER.zip"}
   ,"0804":  {"gid":90804,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0804_HU4_RASTER.zip"}
   ,"0805":  {"gid":90805,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0805_HU4_RASTER.zip"}
   ,"0806":  {"gid":90806,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0806_HU4_RASTER.zip"}
   ,"0807":  {"gid":90807,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0807_HU4_RASTER.zip"}
   ,"0808":  {"gid":90808,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0808_HU4_RASTER.zip"}
   ,"0809":  {"gid":90809,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0809_HU4_RASTER.zip"}
   ,"0901":  {"gid":90901,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0901_HU4_RASTER.zip"}
   ,"0902":  {"gid":90902,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0902_HU4_RASTER.zip"}
   ,"0903":  {"gid":90903,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0903_HU4_RASTER.zip"}
   ,"0904":  {"gid":90904,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_0904_HU4_RASTER.zip"}
   ,"1002":  {"gid":91002,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1002_HU4_RASTER.zip"}
   ,"1003":  {"gid":91003,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1003_HU4_RASTER.zip"}
   ,"1004":  {"gid":91004,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1004_HU4_RASTER.zip"}
   ,"1005":  {"gid":91005,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1005_HU4_RASTER.zip"}
   ,"1006":  {"gid":91006,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1006_HU4_RASTER.zip"}
   ,"1007":  {"gid":91007,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1007_HU4_RASTER.zip"}
   ,"1008":  {"gid":91008,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1008_HU4_RASTER.zip"}
   ,"1009":  {"gid":91009,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1009_HU4_RASTER.zip"}
   ,"1010":  {"gid":91010,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1010_HU4_RASTER.zip"}
   ,"1011":  {"gid":91011,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1011_HU4_RASTER.zip"}
   ,"1012":  {"gid":91012,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1012_HU4_RASTER.zip"}
   ,"1013":  {"gid":91013,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1013_HU4_RASTER.zip"}
   ,"1014":  {"gid":91014,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1014_HU4_RASTER.zip"}
   ,"1015":  {"gid":91015,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1015_HU4_RASTER.zip"}
   ,"1016":  {"gid":91016,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1016_HU4_RASTER.zip"}
   ,"1017":  {"gid":91017,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1017_HU4_RASTER.zip"}
   ,"1018":  {"gid":91018,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1018_HU4_RASTER.zip"}
   ,"1019":  {"gid":91019,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1019_HU4_RASTER.zip"}
   ,"1020":  {"gid":91020,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1020_HU4_RASTER.zip"}
   ,"1021":  {"gid":91021,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1021_HU4_RASTER.zip"}
   ,"1022":  {"gid":91022,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1022_HU4_RASTER.zip"}
   ,"1023":  {"gid":91023,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1023_HU4_RASTER.zip"}
   ,"1024":  {"gid":91024,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1024_HU4_RASTER.zip"}
   ,"1025":  {"gid":91025,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1025_HU4_RASTER.zip"}
   ,"1026":  {"gid":91026,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1026_HU4_RASTER.zip"}
   ,"1027":  {"gid":91027,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1027_HU4_RASTER.zip"}
   ,"1028":  {"gid":91028,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1028_HU4_RASTER.zip"}
   ,"1029":  {"gid":91029,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1029_HU4_RASTER.zip"}
   ,"1030":  {"gid":91030,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1030_HU4_RASTER.zip"}
   ,"1101":  {"gid":91101,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1101_HU4_RASTER.zip"}
   ,"1102":  {"gid":91102,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1102_HU4_RASTER.zip"}
   ,"1103":  {"gid":91103,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1103_HU4_RASTER.zip"}
   ,"1104":  {"gid":91104,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1104_HU4_RASTER.zip"}
   ,"1105":  {"gid":91105,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1105_HU4_RASTER.zip"}
   ,"1106":  {"gid":91106,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1106_HU4_RASTER.zip"}
   ,"1107":  {"gid":91107,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1107_HU4_RASTER.zip"}
   ,"1108":  {"gid":91108,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1108_HU4_RASTER.zip"}
   ,"1109":  {"gid":91109,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1109_HU4_RASTER.zip"}
   ,"1110":  {"gid":91110,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1110_HU4_RASTER.zip"}
   ,"1111":  {"gid":91111,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1111_HU4_RASTER.zip"}
   ,"1112":  {"gid":91112,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1112_HU4_RASTER.zip"}
   ,"1113":  {"gid":91113,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1113_HU4_RASTER.zip"}
   ,"1114":  {"gid":91114,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1114_HU4_RASTER.zip"}
   ,"1201":  {"gid":91201,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1201_HU4_RASTER.zip"}
   ,"1202":  {"gid":91202,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1202_HU4_RASTER.zip"}
   ,"1203":  {"gid":91203,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1203_HU4_RASTER.zip"}
   ,"1204":  {"gid":91204,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1204_HU4_RASTER.zip"}
   ,"1205":  {"gid":91205,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1205_HU4_RASTER.zip"}
   ,"1206":  {"gid":91206,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1206_HU4_RASTER.zip"}
   ,"1207":  {"gid":91207,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1207_HU4_RASTER.zip"}
   ,"1208":  {"gid":91208,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1208_HU4_RASTER.zip"}
   ,"1209":  {"gid":91209,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1209_HU4_RASTER.zip"}
   ,"1210":  {"gid":91210,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1210_HU4_RASTER.zip"}
   ,"1211":  {"gid":91211,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1211_HU4_RASTER.zip"}
   ,"1301":  {"gid":91301,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1301_HU4_RASTER.zip"}
   ,"1302":  {"gid":91302,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1302_HU4_RASTER.zip"}
   ,"1303":  {"gid":91303,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1303_HU4_RASTER.zip"}
   ,"1304":  {"gid":91304,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1304_HU4_RASTER.zip"}
   ,"1305":  {"gid":91305,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1305_HU4_RASTER.zip"}
   ,"1306":  {"gid":91306,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1306_HU4_RASTER.zip"}
   ,"1307":  {"gid":91307,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1307_HU4_RASTER.zip"}
   ,"1308":  {"gid":91308,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1308_HU4_RASTER.zip"}
   ,"1309":  {"gid":91309,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1309_HU4_RASTER.zip"}
   ,"1401":  {"gid":91401,"srid":5070,"priority":10,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1401_HU4_20220414_RASTER.zip"}
   ,"1402":  {"gid":91402,"srid":5070,"priority":10,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1402_HU4_20220414_RASTER.zip"}
   ,"1403":  {"gid":91403,"srid":5070,"priority":10,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1403_HU4_20220414_RASTER.zip"}
   ,"1404":  {"gid":91404,"srid":5070,"priority":10,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1404_HU4_20220414_RASTER.zip"}
   ,"1405":  {"gid":91405,"srid":5070,"priority":10,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1405_HU4_20220414_RASTER.zip"}
   ,"1406":  {"gid":91406,"srid":5070,"priority":10,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1406_HU4_20220414_RASTER.zip"}
   ,"1407":  {"gid":91407,"srid":5070,"priority":10,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1407_HU4_20220414_RASTER.zip"}
   ,"1408":  {"gid":91408,"srid":5070,"priority":10,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1408_HU4_20220414_RASTER.zip"}
   ,"1501":  {"gid":91501,"srid":5070,"priority":10,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1501_HU4_20220901_RASTER.zip"}
   ,"1502":  {"gid":91502,"srid":5070,"priority":10,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1502_HU4_20220901_RASTER.zip"}
   ,"1503":  {"gid":91503,"srid":5070,"priority":10,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1503_HU4_20220901_RASTER.zip"}
   ,"1504":  {"gid":91504,"srid":5070,"priority":10,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1504_HU4_20220901_RASTER.zip"}
   ,"1505":  {"gid":91505,"srid":5070,"priority":10,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1505_HU4_20220901_RASTER.zip"}
   ,"1506":  {"gid":91506,"srid":5070,"priority":10,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1506_HU4_20220901_RASTER.zip"}
   ,"1507":  {"gid":91507,"srid":5070,"priority":10,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1507_HU4_20220901_RASTER.zip"}
   ,"1508":  {"gid":91508,"srid":5070,"priority":10,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1508_HU4_20220826_RASTER.zip"}
   ,"1601":  {"gid":91601,"srid":5070,"priority":10,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1601_HU4_20220412_RASTER.zip"}
   ,"1602":  {"gid":91602,"srid":5070,"priority":10,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1602_HU4_20220412_RASTER.zip"}
   ,"1603":  {"gid":91603,"srid":5070,"priority":10,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1603_HU4_20220412_RASTER.zip"}
   ,"1604":  {"gid":91604,"srid":5070,"priority":10,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1604_HU4_20220418_RASTER.zip"}
   ,"1605":  {"gid":91605,"srid":5070,"priority":10,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1605_HU4_20220418_RASTER.zip"}
   ,"1606":  {"gid":91606,"srid":5070,"priority":10,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1606_HU4_20220418_RASTER.zip"}
   ,"1701":  {"gid":91701,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1701_HU4_RASTER.zip"}
   ,"1702":  {"gid":91702,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1702_HU4_RASTER.zip"}
   ,"1703":  {"gid":91703,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1703_HU4_RASTER.zip"}
   ,"1704":  {"gid":91704,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1704_HU4_RASTER.zip"}
   ,"1705":  {"gid":91705,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1705_HU4_RASTER.zip"}
   ,"1706":  {"gid":91706,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1706_HU4_RASTER.zip"}
   ,"1707":  {"gid":91707,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1707_HU4_RASTER.zip"}
   ,"1708":  {"gid":91708,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1708_HU4_RASTER.zip"}
   ,"1709":  {"gid":91709,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1709_HU4_RASTER.zip"}
   ,"1710":  {"gid":91710,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1710_HU4_RASTER.zip"}
   ,"1711":  {"gid":91711,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1711_HU4_RASTER.zip"}
   ,"1712":  {"gid":91712,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1712_HU4_RASTER.zip"}
   ,"1801":  {"gid":91801,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1801_HU4_RASTER.zip"}
   ,"1802":  {"gid":91802,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1802_HU4_RASTER.zip"}
   ,"1803":  {"gid":91803,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1803_HU4_RASTER.zip"}
   ,"1804":  {"gid":91804,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1804_HU4_RASTER.zip"}
   ,"1805":  {"gid":91805,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1805_HU4_RASTER.zip"}
   ,"1806":  {"gid":91806,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1806_HU4_RASTER.zip"}
   ,"1807":  {"gid":91807,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1807_HU4_RASTER.zip"}
   ,"1808":  {"gid":91808,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1808_HU4_RASTER.zip"}
   ,"1809":  {"gid":91809,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1809_HU4_RASTER.zip"}
   ,"1810":  {"gid":91810,"srid":5070,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_1810_HU4_RASTER.zip"}
   ,"19020101": {"gid":919020101,"srid":3338,"priority":5,"rastertype":"tiff"
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_19020101_HU8_RASTER.zip"}
   ,"19020102": {"gid":919020102,"srid":3338,"priority":5,"rastertype":"tiff"
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_19020102_HU8_RASTER.zip"}
   ,"19020103": {"gid":919020103,"srid":3338,"priority":4,"rastertype":"tiff"
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_19020103_HU8_RASTER.zip","fixsrid":True}
   ,"19020104": {"gid":919020104,"srid":3338,"priority":5,"rastertype":"tiff"
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_19020104_HU8_RASTER.zip"}
   ,"19020201": {"gid":919020201,"srid":3338,"priority":10,"rastertype":"tiff"
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_19020201_HU8_20220713_RASTER.zip"}
   ,"19020202": {"gid":919020202,"srid":3338,"priority":3,"rastertype":"tiff","badproj":5070,"gridsize":5
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_19020202_HU8_RASTER.zip"}
   ,"19020203": {"gid":919020203,"srid":3338,"priority":10,"rastertype":"tiff"
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_19020203_HU8_20220913_RASTER.zip"}
   ,"19020301": {"gid":919020301,"srid":3338,"priority":3,"rastertype":"tiff","badproj":5070,"gridsize":5
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_19020301_HU8_RASTER.zip"}
   ,"19020302": {"gid":919020302,"srid":3338,"priority":3,"rastertype":"tiff","badproj":5070,"gridsize":5
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_19020302_HU8_RASTER.zip"}
   ,"19020401": {"gid":919020401,"srid":3338,"priority":4,"rastertype":"tiff","forcesrid":3338
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_19020401_HU8_RASTER.zip"}
   ,"19020402": {"gid":919020402,"srid":3338,"priority":4,"rastertype":"tiff","forcesrid":3338
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_19020402_HU8_RASTER.zip"}
   ,"19020501": {"gid":919020501,"srid":3338,"priority":4,"rastertype":"tiff","forcesrid":3338
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_19020501_HU8_RASTER.zip"}
   ,"19020502": {"gid":919020502,"srid":3338,"priority":4,"rastertype":"tiff","forcesrid":3338
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_19020502_HU8_RASTER.zip"}
   ,"19020503": {"gid":919020503,"srid":3338,"priority":4,"rastertype":"tiff","forcesrid":3338
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_19020503_HU8_RASTER.zip"}
   ,"19020504": {"gid":919020504,"srid":3338,"priority":4,"rastertype":"tiff","forcesrid":3338
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_19020504_HU8_RASTER.zip"}
   ,"19020505": {"gid":919020505,"srid":3338,"priority":4,"rastertype":"tiff","forcesrid":3338
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_19020505_HU8_RASTER.zip"}
   ,"19020601": {"gid":919020601,"srid":3338,"priority":3,"rastertype":"tiff","badproj":5070,"gridsize":5
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_19020601_HU8_RASTER.zip"}
   ,"19020602": {"gid":919020602,"srid":3338,"priority":3,"rastertype":"tiff","badproj":5070,"gridsize":5
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_19020602_HU8_RASTER.zip"}
   ,"19020800": {"gid":919020800,"srid":3338,"priority":3,"rastertype":"tiff","badproj":5070,"gridsize":5
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_19020800_HU8_RASTER.zip"}
   ,"19050401": {"gid":919050401,"srid":3338,"priority":10,"rastertype":"tiff","forcesrid":3338
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_19050401_HU8_20220711_RASTER.zip"}
   ,"19060102": {"gid":919060102,"srid":3338,"priority":3,"rastertype":"tiff"
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_19060102_HU8_RASTER.zip"}
   ,"19060501": {"gid":919060501,"srid":3338,"priority":10,"rastertype":"tiff"
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_19060501_HU8_20220712_RASTER.zip"}
   ,"19060502": {"gid":919060502,"srid":3338,"priority":10,"rastertype":"tiff"
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_19060502_HU8_20220713_RASTER.zip"}
   ,"19060504": {"gid":919060504,"srid":3338,"priority":10,"rastertype":"tiff"
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_19060504_HU8_20220826_RASTER.zip"}
   ,"19060505": {"gid":919060505,"srid":3338,"priority":2,"rastertype":"tiff","badproj":5070,"gridsize":5
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_19060505_HU8_20220927_RASTER.zip"}
   ,"19070402": {"gid":919070402,"srid":3338,"priority":3,"rastertype":"tiff"
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_19070402_HU8_RASTER.zip"}
   ,"19080301": {"gid":919080301,"srid":3338,"priority":3,"rastertype":"tiff"
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_19080301_HU8_RASTER.zip"}
   ,"19080305": {"gid":919080305,"srid":3338,"priority":3,"rastertype":"tiff"
                 ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_19080305_HU8_RASTER.zip"}
   ,"2001":  {"gid":92001,"srid":26904,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_2001_HU4_RASTER.zip"}
   ,"2002":  {"gid":92002,"srid":26904,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_2002_HU4_RASTER.zip"}
   ,"2003":  {"gid":92003,"srid":26904,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_2003_HU4_RASTER.zip"}
   ,"2004":  {"gid":92004,"srid":26904,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_2004_HU4_RASTER.zip"}
   ,"2005":  {"gid":92005,"srid":26904,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_2005_HU4_RASTER.zip"}
   ,"2006":  {"gid":92006,"srid":26904,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_2006_HU4_RASTER.zip"}
   ,"2007":  {"gid":92007,"srid":26904,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_2007_HU4_RASTER.zip"}
   ,"2008":  {"gid":92008,"srid":26904,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_2008_HU4_RASTER.zip"}
   ,"2101":  {"gid":92101,"srid":32161,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_2101_HU4_RASTER.zip"}
   ,"2102":  {"gid":92102,"srid":32161,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_2102_HU4_RASTER.zip"}
   ,"2201":  {"gid":92201,"srid":32655,"priority":3,"rastertype":"tiff","badproj":32655,"gridsize":10.233
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_2201_HU4_RASTER.zip"}
   ,"2202":  {"gid":92202,"srid":32655,"priority":3,"rastertype":"tiff","badproj":32655,"gridsize":10.233
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_2202_HU4_RASTER.zip"}
   ,"2203":  {"gid":92203,"srid":32702,"priority":3,"rastertype":"tiff"
              ,"url":"https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/NHDPlusHR/VPU/Current/Raster/NHDPLUS_H_2203_HU4_RASTER.zip"}
};

if check_hr_r2_links is True:
    for k,v in hr_r2_grids.items():
        check_link(v['url']);

if dwnld_hr_r2_links is True:
    for k,v in hr_r2_grids.items():
        download_link(
             v['url']
            ,os.path.join(lddk,'hr_r2',os.path.basename(v['url']))
        );


In [ ]:
wrk_set       = 'hr_r1';
wrk_grid_size = hr_r1_grid_size;
wrk_grids     = hr_r1_grids;

In [ ]:
unzip_files = False;

if unzip_files is True:

    for srid in [26904,32161,32655,32702,3338,5070]:
        outdir = os.path.join(lddk,wrk_set,'fdr_' + str(srid));
        if not os.path.exists(outdir):
            os.mkdir(outdir);
    
    tiffiles = ['fdr.tfw','fdr.tif','fdr.tif.aux.xml','fdr.tif.xml'];
    
    for k,v in wrk_grids.items():
        zf = os.path.join(lddk,wrk_set,os.path.basename(v['url']));
        ex = os.path.splitext(v['url'])[1];
        trgdir = os.path.join(lddk,wrk_set,'fdr_' + str(v['srid']));
        
        if v['rastertype'] == 'tiff':
        
            if ex == '.zip':
                with zipfile.ZipFile(zf,'r') as z:
                    filelst = [name for name in z.namelist() if not name.endswith('/')];
                    
                    for file in tiffiles:
            
                        src = None;
                        for f in filelst:
                            if os.path.basename(f) == file:
                                src = f;
                                break;
            
                        if src is None:
                            print('unable to find ' + file + ' in ' + z);
                            #raise Exception('foo')
                        
                        info = z.getinfo(src);
                        info.filename = os.path.basename(info.filename.replace('fdr.','fdr_' + k + '.'));
                        #print(info.filename);
            
                        z.extract(info,path = os.path.join(lddk,wrk_set,'fdr_' + str(v['srid'])));
        
            # admin container includes py7zip but its super slow
            elif ex == '.7z':
                z = common.p7zip('e -o' + trgdir + ' ' + zf + ' ' + ' '.join(tiffiles) + ' -r');
                print(str(z));
                
                for file in tiffiles:
                    os.rename(
                         os.path.join(trgdir,file)
                        ,os.path.join(trgdir,file.replace('fdr.','fdr_' + k + '.'))
                    );

        elif v['rastertype'] == 'esrigrid':
            #print(v['url']);
            if os.path.exists('/tmp/fdr'):
                shutil.rmtree('/tmp/fdr');
            
            z = common.p7zip('e -aoa -o/tmp/fdr ' + zf + ' fdr -r');

            outtif = os.path.join(trgdir,'fdr_' + k + '.tif');
            z = common.gdal_translate('-of GTiff -stats -co TFW=YES -co COMPRESS=LZW /tmp/fdr ' + outtif);
            print(str(z))
            
        else:
            raise Exception('unknown raster type');
        

In [ ]:
fix_badproj = False;

if fix_badproj is True:

    for k,v in wrk_grids.items():

        if 1==1:
            
            if 'badproj' in v:
                intif  = os.path.join(lddk,wrk_set,'fdr_' + str(v['srid']),'fdr_' + str(k) + '.tif');
                outtif = os.path.join(lddk,wrk_set,'fdr_' + str(v['srid']),'fdr_' + str(k) + 'fixed.tif');

                print(' warping ' + intif + ' to ' + str(v['srid']));
                z = common.gdalwarp('-r near -tr ' + str(v['gridsize']) + ' ' + str(v['gridsize']) + ' -t_srs EPSG:' + str(v['srid']) + ' ' + intif + ' ' +  outtif);
    

In [ ]:
# note logic does not allow same raster to be both in fix_badproj and force_srid status
force_srid = True;

if force_srid is True:

    for k,v in wrk_grids.items():

        if 1==1:

            if 'forcesrid' in v:
                intif  = os.path.join(lddk,wrk_set,'fdr_' + str(v['srid']),'fdr_' + str(k) + '.tif');
                outtif = os.path.join(lddk,wrk_set,'fdr_' + str(v['srid']),'fdr_' + str(k) + 'fixed.tif');

                print(' forcing ' + intif + ' to ' + str(v['forcesrid']));
                if os.path.exists(outtif):
                    os.remove(outtif);
                    
                shutil.copy(intif,outtif);
                z = common.gdal_edit('-a_srs EPSG:' + str(v['forcesrid']) + ' ' + outtif);
                

In [ ]:
build_footprints = False;

if build_footprints is True:
    if os.path.exists('/tmp/tmp.tif'):
        os.remove('/tmp/tmp.tif');
        
    ftprt = {};

    for srid in [26904,32161,32655,32702,3338,5070]:
        ftprtgpkg = os.path.join(lddk,wrk_set,'fdr_' + str(srid),'fdr_' + str(srid) + '_footprint.gpkg');
        if os.path.exists(ftprtgpkg):
            os.remove(ftprtgpkg);
                
        ftprt[srid] = ftprtgpkg;

    for k,v in wrk_grids.items():
        
        if 1==1:
            fxd = os.path.join(lddk,wrk_set,'fdr_' + str(v['srid']),'fdr_' + str(k) + 'fixed.tif');
            if os.path.exists(fxd):
                tif = fxd;
            else:
                tif = os.path.join(lddk,wrk_set,'fdr_' + str(v['srid']),'fdr_' + str(k) + '.tif');
            nfo = os.path.join(lddk,wrk_set,'fdr_' + str(v['srid']),'fdr_' + str(k) + '.gdalinfo');
    
            with open(nfo,"wb") as f:
                stdout_data, stderr_data = common.gdalinfo(tif);
                f.write(stdout_data);
    
            print('creating temp byte mask for raster' + tif);
            z = common.gdal_calc('-A ' + tif + ' --outfile=/tmp/tmp.tif --NoDataValue=0 --calc="where(A<255,1,0)" --type=Byte --overwrite');
            print(z);
            print('polygonizing result');
            z = common.gdal_polygonize('-8 /tmp/tmp.tif ' + ftprt[v['srid']] + ' footprints gid');
            print(z);
            z = common.ogrinfo(ftprt[v['srid']] + ' -sql "UPDATE main.footprints SET gid = ' + str(v['gid']) + ' WHERE gid = 1"');
            print(z);
            
            if os.path.exists('/tmp/tmp.tif'):
                os.remove('/tmp/tmp.tif');
        

In [ ]:
build_vrt = False;

if build_vrt is True:
    
    for srid in [26904,32161,32655,32702,3338,5070]:
        trgdir = os.path.join(lddk,wrk_set,'fdr_' + str(srid));
        vrtfil = os.path.join(trgdir,'fdr_' + str(srid) + '.vrt');
        tiflst = os.path.join(trgdir,'fdr_' + str(srid) + '_list.txt');
        
        srid_dict = {k:v for k,v in wrk_grids.items() if v['srid'] == srid}
        srtd_dict = dict(sorted(srid_dict.items(),key=lambda item: item[1]['priority']));
    
        with open(tiflst,"w") as f:
            for k,v in srtd_dict.items():
                fxd = os.path.join(lddk,wrk_set,'fdr_' + str(v['srid']),'fdr_' + str(k) + 'fixed.tif');
                if os.path.exists(fxd):
                    tif = fxd;
                else:
                    tif = os.path.join(lddk,wrk_set,'fdr_' + str(v['srid']),'fdr_' + str(k) + '.tif');
                
                f.write(tif + '\n');
    
        z = common.gdalbuildvrt(vrtfil + ' -input_file_list ' + tiflst + ' -tr ' + str(wrk_grid_size[srid]) + ' ' + str(wrk_grid_size[srid]))
        print(str(z));
    

In [ ]:
mosaick_rasters = False;

if mosaick_rasters is True:

    for srid in [26904,32161,32655,32702,3338,5070]:
        trgdir = os.path.join(lddk,wrk_set,'fdr_' + str(srid));
        vrtfil = os.path.join(trgdir,'fdr_' + str(srid) + '.vrt');
        